In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
df = pd.read_csv("amz_uk_price_prediction_dataset.csv")

### Part 1: Analyzing Best-Seller Trends Across Product Categories

**Objective**: Understand the relationship between product categories and their best-seller status.

1. **Crosstab Analysis**:
    - Create a crosstab between the product `category` and the `isBestSeller` status.
    
    - Are there categories where being a best-seller is more prevalent? 
    	
    	*Hint: one option is to calculate the proportion of best-sellers for each category and then sort the categories based on this proportion in descending order.*


2. **Statistical Tests**:
    - Conduct a Chi-square test to determine if the best-seller distribution is independent of the product category.
    - Compute Cramér's V to understand the strength of association between best-seller status and category.

3. **Visualizations**:
	- Visualize the relationship between product categories and the best-seller status using a stacked bar chart.

In [9]:
best_seller_crosstab = pd.crosstab(df["category"], df["isBestSeller"])
best_seller_crosstab = best_seller_crosstab.rename(columns={False: "not_best_seller", True: "best_seller"})

# Ensure both columns exist even if a future dataset version lacks one status.
for column in ["not_best_seller", "best_seller"]:
    if column not in best_seller_crosstab.columns:
        best_seller_crosstab[column] = 0

best_seller_crosstab = best_seller_crosstab[["not_best_seller", "best_seller"]]
best_seller_crosstab["total_listings"] = best_seller_crosstab.sum(axis=1)
best_seller_crosstab["best_seller_pct"] = (
    best_seller_crosstab["best_seller"] / best_seller_crosstab["total_listings"] * 100
)

best_seller_ranked = best_seller_crosstab.sort_values(
    ["best_seller_pct", "total_listings"],
    ascending=[False, False]
)

display(best_seller_ranked.head(20))

min_listing_count = 1000
best_seller_ranked_reliable = best_seller_ranked[
    best_seller_ranked["total_listings"] >= min_listing_count
]
display(best_seller_ranked_reliable.head(20))

isBestSeller,not_best_seller,best_seller,total_listings,best_seller_pct
category,,,,
Grocery,9008,556,9564,5.813467
Smart Home Security & Lighting,98,6,104,5.769231
Health & Personal Care,9017,552,9569,5.768628
Mobile Phone Accessories,248,11,259,4.247104
Power & Hand Tools,8353,306,8659,3.533895
"Billiard, Snooker & Pool",241,8,249,3.212851
Pet Supplies,9152,285,9437,3.020028
Home Brewing & Wine Making,239,7,246,2.845528
Mirrors,243,7,250,2.800000


isBestSeller,not_best_seller,best_seller,total_listings,best_seller_pct
category,,,,
Grocery,9008,556,9564,5.813467
Health & Personal Care,9017,552,9569,5.768628
Power & Hand Tools,8353,306,8659,3.533895
Pet Supplies,9152,285,9437,3.020028
PC & Video Games,6008,157,6165,2.546634
Arts & Crafts,9319,230,9549,2.408629
Baby,14709,341,15050,2.265781
Large Appliances,7740,144,7884,1.826484
Hardware,8424,146,8570,1.703617


In [10]:
def chi_square_independence(observed_table):
    observed = observed_table.to_numpy(dtype=float)
    row_totals = observed.sum(axis=1)
    col_totals = observed.sum(axis=0)
    n = observed.sum()
    expected = np.outer(row_totals, col_totals) / n

    chi2 = np.divide(
        (observed - expected) ** 2,
        expected,
        out=np.zeros_like(expected),
        where=expected != 0
    ).sum()
    dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
    cramers_v = np.sqrt(chi2 / (n * min(observed.shape[0] - 1, observed.shape[1] - 1)))

    # Wilson-Hilferty normal approximation for the chi-square survival probability.
    # Here the test statistic is so large that the p-value underflows to 0.0.
    if dof > 0 and chi2 > 0:
        z = ((chi2 / dof) ** (1 / 3) - (1 - 2 / (9 * dof))) / np.sqrt(2 / (9 * dof))
        p_value_approx = max(0.0, 1 - NormalDist().cdf(z))
    else:
        z = np.nan
        p_value_approx = np.nan

    return {
        "chi_square": chi2,
        "degrees_of_freedom": dof,
        "p_value_approx": p_value_approx,
        "wilson_hilferty_z": z,
        "cramers_v": cramers_v
    }

chi_square_results = chi_square_independence(best_seller_crosstab[["not_best_seller", "best_seller"]])
display(pd.DataFrame([chi_square_results]))

NameError: name 'NormalDist' is not defined

### Part 2: Exploring Product Prices and Ratings Across Categories and Brands

**Objective**: Investigate how different product categories influence product prices.

0. **Preliminary Step: Remove outliers in product prices.**

	For this purpose, we can use the IQR (Interquartile Range) method. Products priced below the first quartile minus 1.5 times the IQR or above the third quartile plus 1.5 times the IQR will be considered outliers and removed from the dataset. The next steps will be done with the dataframe without outliers.
	
	*Hint: you can check the last Check For Understanding at the end of the lesson EDA Bivariate Analysis for a hint on how to do this.*

1. **Violin Plots**:
    - Use a violin plot to visualize the distribution of `price` across different product `categories`. Filter out the top 20 categories based on count for better visualization.
    - Which product category tends to have the highest median price? Don't filter here by top categories.

2. **Bar Charts**:
    - Create a bar chart comparing the average price of products for the top 10 product categories (based on count).
    - Which product category commands the highest average price? Don't filter here by top categories.

3. **Box Plots**:
    - Visualize the distribution of product `ratings` based on their `category` using side-by-side box plots. Filter out the top 10 categories based on count for better visualization.
    - Which category tends to receive the highest median rating from customers? Don't filter here by top categories.

### Part 3: Investigating the Interplay Between Product Prices and Ratings

**Objective**: Analyze how product ratings (`stars`) correlate with product prices.

1. **Correlation Coefficients**:
    - Calculate the correlation coefficient between `price` and `stars`.
    - Is there a significant correlation between product price and its rating?
	
2. **Visualizations**:
    - Use a scatter plot to visualize the relationship between product rating and price. What patterns can you observe?
    - Use a correlation heatmap to visualize correlations between all numerical variables.
    - Examine if product prices typically follow a normal distribution using a QQ plot. 